# A-Share Trend Strategy - Advanced Technical Analysis

This notebook demonstrates the usage of the `DualMAOptimizedStrategy` class, which implements an advanced dual moving average strategy with comprehensive technical indicators and risk management for A-share markets.

## Features

- **Technical Indicators**: Moving Averages, RSI, MACD, Bollinger Bands, ADX, OBV, VWAP
- **Risk Management**: Fixed stop-loss, ATR trailing stop, time-based stops
- **Position Sizing**: Pyramid entries, capital allocation limits
- **Volume Analysis**: Volume spike detection, OBV trends

In [ ]:
# Import required libraries
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Add src directory to path
sys.path.append('../src')

from optimized_strategy import DualMAOptimizedStrategy, create_realistic_sample_data

## 1. Create Sample Data

Let's generate realistic market data with trending patterns that will trigger our trading signals.

In [ ]:
# Create realistic sample data
sample_data = create_realistic_sample_data(
    start_date='2023-01-01', 
    end_date='2023-12-31', 
    initial_price=100
)

print(f"Sample data shape: {sample_data.shape}")
print(f"Date range: {sample_data.index[0]} to {sample_data.index[-1]}")
sample_data.head()

## 2. Visualize Raw Data

Let's plot the price and volume data to understand the market patterns.

In [ ]:
# Plot price and volume data
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10))

# Price chart
ax1.plot(sample_data.index, sample_data['close'], label='Close Price', linewidth=1)
ax1.set_title('Stock Price Over Time')
ax1.set_ylabel('Price')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Volume chart
ax2.bar(sample_data.index, sample_data['volume'], alpha=0.7, color='orange')
ax2.set_title('Trading Volume Over Time')
ax2.set_ylabel('Volume')
ax2.set_xlabel('Date')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Initialize Strategy

Create the strategy instance with the required parameters for A-share markets.

In [ ]:
# Initialize strategy with A-share optimized parameters
strategy = DualMAOptimizedStrategy(
    fast_ma_period=8,           # Fast MA: 8 days
    slow_ma_period=21,          # Slow MA: 21 days  
    adx_period=14,              # ADX: 14 days
    rsi_period=14,              # RSI: 14 days
    rsi_overbought=70,          # RSI overbought threshold
    rsi_oversold=30,            # RSI oversold threshold
    bb_period=20,               # Bollinger Bands: 20 days
    bb_std=2.0,                 # Bollinger Bands: ±2 STD
    volume_spike_multiplier=1.5, # Volume spike: >1.5x average
    stop_loss_pct=0.08,         # Fixed stop loss: 8%
    atr_multiplier=3.0,         # ATR trailing stop: 3x ATR
    time_stop_days=20,          # Time stop: 20 days
    max_positions=3,            # Pyramid entries: up to 3
    max_capital_per_stock=0.30  # Max capital per stock: 30%
)

print("Strategy initialized with A-share optimized parameters")
print(f"Fast MA Period: {strategy.fast_ma_period} days")
print(f"Slow MA Period: {strategy.slow_ma_period} days")
print(f"Stop Loss: {strategy.stop_loss_pct:.1%}")
print(f"Time Stop: {strategy.time_stop_days} days")

## 4. Calculate Technical Indicators

Calculate all technical indicators and examine their values.

In [ ]:
# Calculate all technical indicators
data_with_indicators = strategy.calculate_all_indicators(sample_data)

print("Technical indicators calculated:")
print(f"Data shape: {data_with_indicators.shape}")
print("\nIndicator columns:")
indicator_columns = [col for col in data_with_indicators.columns if col not in ['open', 'high', 'low', 'close', 'volume']]
for col in indicator_columns:
    print(f"  - {col}")

# Display sample of indicators
print("\nSample indicator values:")
display_cols = ['close', 'ma_fast', 'ma_slow', 'rsi', 'macd_histogram', 'bb_upper', 'bb_lower', 'volume_spike']
data_with_indicators[display_cols].tail(10)

## 5. Generate Trading Signals

Generate buy and sell signals based on the technical indicators.

In [ ]:
# Generate trading signals
data_with_signals = strategy.generate_signals(data_with_indicators)

# Count signals
buy_signals = data_with_signals['buy_signal'].sum()
sell_signals = data_with_signals['sell_signal'].sum()

print(f"Total Buy Signals: {buy_signals}")
print(f"Total Sell Signals: {sell_signals}")

# Show signal dates
buy_dates = data_with_signals[data_with_signals['buy_signal'] == True].index
sell_dates = data_with_signals[data_with_signals['sell_signal'] == True].index

if len(buy_dates) > 0:
    print(f"\nFirst 5 Buy Signal Dates:")
    for date in buy_dates[:5]:
        print(f"  {date.strftime('%Y-%m-%d')}")

if len(sell_dates) > 0:
    print(f"\nFirst 5 Sell Signal Dates:")
    for date in sell_dates[:5]:
        print(f"  {date.strftime('%Y-%m-%d')}")

## 6. Visualize Technical Indicators

Plot the key technical indicators with buy/sell signals.

In [ ]:
# Create comprehensive technical analysis chart
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 12))

# Plot 1: Price with Moving Averages and Bollinger Bands
ax1.plot(data_with_signals.index, data_with_signals['close'], label='Close Price', linewidth=1)
ax1.plot(data_with_signals.index, data_with_signals['ma_fast'], label=f'MA Fast ({strategy.fast_ma_period})', alpha=0.8)
ax1.plot(data_with_signals.index, data_with_signals['ma_slow'], label=f'MA Slow ({strategy.slow_ma_period})', alpha=0.8)
ax1.plot(data_with_signals.index, data_with_signals['bb_upper'], label='BB Upper', linestyle='--', alpha=0.6)
ax1.plot(data_with_signals.index, data_with_signals['bb_lower'], label='BB Lower', linestyle='--', alpha=0.6)
ax1.fill_between(data_with_signals.index, data_with_signals['bb_upper'], data_with_signals['bb_lower'], alpha=0.1)

# Add buy/sell signals
buy_points = data_with_signals[data_with_signals['buy_signal'] == True]
sell_points = data_with_signals[data_with_signals['sell_signal'] == True]
ax1.scatter(buy_points.index, buy_points['close'], color='green', marker='^', s=100, label='Buy Signal', zorder=5)
ax1.scatter(sell_points.index, sell_points['close'], color='red', marker='v', s=100, label='Sell Signal', zorder=5)

ax1.set_title('Price Chart with Moving Averages, Bollinger Bands & Signals')
ax1.set_ylabel('Price')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: RSI
ax2.plot(data_with_signals.index, data_with_signals['rsi'], label='RSI', color='purple')
ax2.axhline(y=strategy.rsi_overbought, color='red', linestyle='--', alpha=0.7, label=f'Overbought ({strategy.rsi_overbought})')
ax2.axhline(y=strategy.rsi_oversold, color='green', linestyle='--', alpha=0.7, label=f'Oversold ({strategy.rsi_oversold})')
ax2.axhline(y=50, color='gray', linestyle='-', alpha=0.5)
ax2.set_title('Relative Strength Index (RSI)')
ax2.set_ylabel('RSI')
ax2.set_ylim(0, 100)
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: MACD
ax3.plot(data_with_signals.index, data_with_signals['macd_line'], label='MACD Line', color='blue')
ax3.plot(data_with_signals.index, data_with_signals['macd_signal'], label='Signal Line', color='red')
ax3.bar(data_with_signals.index, data_with_signals['macd_histogram'], label='Histogram', alpha=0.7, color='gray')
ax3.axhline(y=0, color='black', linestyle='-', alpha=0.5)
ax3.set_title('MACD (Moving Average Convergence Divergence)')
ax3.set_ylabel('MACD')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Volume with Spikes
colors = ['red' if spike else 'blue' for spike in data_with_signals['volume_spike']]
ax4.bar(data_with_signals.index, data_with_signals['volume'], alpha=0.7, color=colors)
ax4.plot(data_with_signals.index, data_with_signals['volume_avg'], label='Volume Average', color='orange', linewidth=2)
ax4.set_title('Volume Analysis (Red = Volume Spikes)')
ax4.set_ylabel('Volume')
ax4.set_xlabel('Date')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Run Backtest

Execute the complete backtest and analyze the results.

In [ ]:
# Run the backtest
results = strategy.backtest_strategy(sample_data)

print("=== Dual MA Optimized Strategy Backtest Results ===")
print(f"Total Trades: {results['total_trades']}")
print(f"Total Return: {results['total_return']:.2%}")
print(f"Win Rate: {results['win_rate']:.2%}")
print(f"Average Return per Trade: {results['avg_return']:.2%}")
print(f"Best Trade: {results['max_return']:.2%}")
print(f"Worst Trade: {results['min_return']:.2%}")

if results['positions']:
    print(f"\n=== Trade Details ===")
    positions_df = pd.DataFrame(results['positions'])
    print(positions_df.to_string(index=False))
else:
    print("\nNo trades were generated with the current parameters and data.")
    print("This could be due to strict entry conditions. Consider:")
    print("1. Adjusting RSI thresholds")
    print("2. Lowering ADX threshold for trend strength")
    print("3. Reducing volume spike requirements")

## 8. Risk Management Analysis

Examine the risk management features in detail.

In [ ]:
# Demonstrate risk management calculations
if len(data_with_indicators) > 50:
    # Example position entry
    entry_date = data_with_indicators.index[30]
    entry_price = data_with_indicators.loc[entry_date, 'close']
    
    # Calculate stop levels for multiple dates
    print(f"=== Risk Management Example ===")
    print(f"Entry Date: {entry_date.strftime('%Y-%m-%d')}")
    print(f"Entry Price: ${entry_price:.2f}")
    print()
    
    # Check stop levels at different points
    check_dates = [40, 45, 50]
    
    for day_offset in check_dates:
        if day_offset < len(data_with_indicators):
            check_date = data_with_indicators.index[day_offset]
            current_data = data_with_indicators.iloc[:day_offset+1]
            
            stop_levels = strategy.calculate_stop_levels(current_data, entry_price, entry_date)
            
            if stop_levels:
                print(f"Day {day_offset - 30} after entry ({check_date.strftime('%Y-%m-%d')}):")
                print(f"  Current Price: ${stop_levels['current_price']:.2f}")
                print(f"  Fixed Stop: ${stop_levels['fixed_stop']:.2f} ({strategy.stop_loss_pct:.1%} loss)")
                print(f"  ATR Stop: ${stop_levels['atr_stop']:.2f}")
                print(f"  Days Held: {stop_levels['days_held']}")
                print(f"  Time Stop Triggered: {stop_levels['time_stop_triggered']}")
                
                should_exit, reason = strategy.should_exit_position(current_data, entry_price, entry_date)
                print(f"  Should Exit: {should_exit} ({reason})")
                print()
else:
    print("Not enough data for risk management demonstration")

## 9. Parameter Sensitivity Analysis

Test how different parameters affect the strategy performance.

In [ ]:
# Test different parameter combinations
param_tests = [
    {"name": "Conservative", "rsi_overbought": 80, "rsi_oversold": 20, "volume_spike_multiplier": 2.0},
    {"name": "Aggressive", "rsi_overbought": 60, "rsi_oversold": 40, "volume_spike_multiplier": 1.2},
    {"name": "Original", "rsi_overbought": 70, "rsi_oversold": 30, "volume_spike_multiplier": 1.5}
]

print("=== Parameter Sensitivity Analysis ===")
print(f"{'Strategy':<12} {'Trades':<8} {'Total Return':<12} {'Win Rate':<10} {'Avg Return':<12}")
print("-" * 60)

for params in param_tests:
    test_strategy = DualMAOptimizedStrategy(
        fast_ma_period=8,
        slow_ma_period=21,
        rsi_overbought=params["rsi_overbought"],
        rsi_oversold=params["rsi_oversold"],
        volume_spike_multiplier=params["volume_spike_multiplier"]
    )
    
    test_results = test_strategy.backtest_strategy(sample_data)
    
    print(f"{params['name']:<12} {test_results['total_trades']:<8} "
          f"{test_results['total_return']:<12.2%} {test_results['win_rate']:<10.2%} "
          f"{test_results['avg_return']:<12.2%}")

## 10. Strategy Summary

### Key Features Implemented:

1. **Technical Indicators**:
   - Moving Averages (8-day fast, 21-day slow)
   - RSI with 70/30 thresholds
   - MACD with histogram crossover
   - Bollinger Bands (20-day ± 2 STD)
   - ADX for trend strength
   - OBV and VWAP for volume analysis

2. **Risk Management**:
   - Fixed 8% stop loss
   - ATR-based trailing stop (3× ATR)
   - 20-day maximum holding period

3. **Position Sizing**:
   - Up to 3 pyramid entries
   - Maximum 30% capital per stock
   - Volume spike confirmation (>1.5× average)

### Usage in Production:

1. Replace sample data with real market data
2. Adjust parameters based on market conditions
3. Implement proper position sizing based on account size
4. Add sector diversification controls
5. Include transaction costs and slippage

### Next Steps:

- Backtest on historical A-share data
- Optimize parameters using walk-forward analysis
- Implement live trading integration
- Add performance analytics and reporting